# S03 · Your first DataFrame — a spreadsheet you drive with code

We take the shop owner's sales and put them in a table you control with code. We
build the table by hand, look at its rows and columns, pull out the parts we want,
and keep only the rows that pass a yes/no test.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on
  each cell, top to bottom, and read the plain-English note above each one.
- Think of a DataFrame as an Excel sheet. Every move below is something you have
  probably done by clicking in a spreadsheet; here you type it instead.
- Already comfortable with pandas or with Excel formulas? Skip to the cell marked
  **Stretch (optional)** near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [1]:
# This notebook only uses pandas, which Google Colab already ships.
# So there is nothing to install here.
print("Setup complete - nothing to install.")

Setup complete - nothing to install.


In [2]:
import pandas as pd   # pandas is the library for working with tables

## Step 1 — read the shop's sales file

A **DataFrame** is just a table: rows and columns, like a spreadsheet. Instead
of typing the numbers in by hand, we read them straight from a file — exactly
what you do at work when someone hands you a spreadsheet.

The file `shop_sales.csv` ships with this session (in the `data/` folder). A CSV
is plain text with one row per line and commas between the columns. `pd.read_csv`
turns it into a table in a single line.

In [3]:
from pathlib import Path

# Read shop_sales.csv three ways, so this notebook runs anywhere:
#   1) from the repo's data/ folder (you're inside the course repo),
#   2) else download it from the course website (e.g. on Google Colab),
#   3) else rebuild it in memory as a last resort.
csv_path = Path("../data/shop_sales.csv")
DATA_URL = "https://girishmkulkarni.github.io/applied-maths-in-industry-site/sessions/S03/data/shop_sales.csv"

if csv_path.exists():
    sales = pd.read_csv(csv_path)
else:
    try:
        sales = pd.read_csv(DATA_URL)
    except Exception:
        sales = pd.DataFrame({
            "date": ["2024-01-03", "2024-01-05", "2024-01-06", "2024-01-08",
                     "2024-01-09", "2024-01-11", "2024-01-12", "2024-01-14",
                     "2024-01-15", "2024-01-17", "2024-01-18", "2024-01-20"],
            "city": ["Pune", "Mumbai", "Pune", "Delhi", "Mumbai", "Pune",
                     "Delhi", "Mumbai", "Pune", "Mumbai", "Delhi", "Pune"],
            "customer": ["Asha", "Ravi", "Meera", "John", "Sara", "Asha",
                         "Vikram", "Ravi", "Kiran", "Sara", "John", "Meera"],
            "units": [10, 4, 7, 2, 9, 5, 6, 8, 3, 13, 4, 6],
            "price": [250, 300, 250, 180, 300, 250, 180, 300, 250, 300, 180, 250],
        })

# Look at the first few rows.
print(sales.head())

         date    city customer  units  price
0  2024-01-03    Pune     Asha     10    250
1  2024-01-05  Mumbai     Ravi      4    300
2  2024-01-06    Pune    Meera      7    250
3  2024-01-08   Delhi     John      2    180
4  2024-01-09  Mumbai     Sara      9    300


## Step 2 — look before you compute

The first thing to do with any new file is *look at it*. Every DataFrame has an
**index** (the row labels down the left, here the numbers 0, 1, 2, …) and a set
of **column** names across the top. `shape` tells you (rows, columns), and
`info()` shows every column's type and how many values are present — your first
line of defence against a messy file.

In [4]:
# The column names of the table.
print("columns:", list(sales.columns))

# How many rows and columns? The answer is (rows, columns).
print("shape  :", sales.shape)

# The full summary: column types and how many values are present.
print()
sales.info()

columns: ['date', 'city', 'customer', 'units', 'price']
shape  : (12, 5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   date      12 non-null     object
 1   city      12 non-null     object
 2   customer  12 non-null     object
 3   units     12 non-null     int64 
 4   price     12 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 612.0+ bytes


### Your turn (2 min)

You now have the file loaded. Answer these from the table:

1. How many rows and columns are there? (hint: `sales.shape`)
2. Which cities appear in the file? (hint: `sales["city"].unique()`)
3. Which city sold the most units? Eyeball it now — you'll nail it in one line
   with `groupby` in the next notebook.

In [5]:
# 1. rows and columns
print("rows, columns:", sales.shape)

# 2. the distinct cities
print("cities:", sales["city"].unique())

# 3. your guess for the top city by units — check it with groupby in notebook 02

rows, columns: (12, 5)
cities: ['Pune' 'Mumbai' 'Delhi']


## Step 3 — pick out one column

To get a single column, write the table's name and the column name in square
brackets. One column on its own is called a **Series**. It still carries the row
labels, so you always know which value belongs to which row.

In [6]:
# Get just the "units" column.
units_column = sales["units"]

print(units_column)

0     10
1      4
2      7
3      2
4      9
5      5
6      6
7      8
8      3
9     13
10     4
11     6
Name: units, dtype: int64


## Step 4 — pick rows and columns by name with `.loc`

`.loc` selects using **names**: the row's index label and the column's name. The
pattern is `sales.loc[rows, columns]`. This is the everyday way to reach into a
table when your rows and columns have meaningful labels.

In [7]:
# The single value in row 0, column "customer".
print("row 0, customer:", sales.loc[0, "customer"])

# Row 2, but only the columns "customer" and "city".
print()
print("row 2, two columns:")
print(sales.loc[2, ["customer", "city"]])

row 0, customer: Asha

row 2, two columns:
customer    Meera
city         Pune
Name: 2, dtype: object


## Step 5 — pick rows and columns by position with `.iloc`

`.iloc` selects using **positions**, counting from 0, instead of names. This is
handy when you just want "the first few rows" and do not care what they are called.
The `i` in `iloc` is a reminder that it works by integer position.

In [8]:
# The first row (position 0). Counting starts at 0.
print("first row:")
print(sales.iloc[0])

# The first three rows (positions 0, 1, 2). The 3 is not included.
print()
print("first three rows:")
print(sales.iloc[0:3])

first row:
date        2024-01-03
city              Pune
customer          Asha
units               10
price              250
Name: 0, dtype: object

first three rows:
         date    city customer  units  price
0  2024-01-03    Pune     Asha     10    250
1  2024-01-05  Mumbai     Ravi      4    300
2  2024-01-06    Pune    Meera      7    250


## Step 6 — keep only the rows you want (a boolean mask)

A very common task: keep only the rows that pass a test. This is exactly the shop
owner asking "show me just the big orders". First we make a column of True/False
values (a **boolean mask**), then we use it to keep the True rows. Here we keep
customers who bought more than 5 units.

In [9]:
# Step A: build the mask. For each row, is "units" greater than 5?
bought_more_than_5 = sales["units"] > 5

print("the mask (True means keep this row):")
print(bought_more_than_5)

the mask (True means keep this row):
0      True
1     False
2      True
3     False
4      True
5     False
6      True
7      True
8     False
9      True
10    False
11     True
Name: units, dtype: bool


Now put that mask in square brackets to keep only the rows where it is True.

In [10]:
# Step B: index with the mask to keep only the True rows.
big_buyers = sales[bought_more_than_5]

print("customers who bought more than 5 units:")
print(big_buyers)

customers who bought more than 5 units:
          date    city customer  units  price
0   2024-01-03    Pune     Asha     10    250
2   2024-01-06    Pune    Meera      7    250
4   2024-01-09  Mumbai     Sara      9    300
6   2024-01-12   Delhi   Vikram      6    180
7   2024-01-14  Mumbai     Ravi      8    300
9   2024-01-17  Mumbai     Sara     13    300
11  2024-01-20    Pune    Meera      6    250


## Step 7 — count rows per category with `value_counts`

`value_counts()` tallies how many rows fall in each category — here, how many
sales came from each city. Add `normalize=True` to get the share instead of the
raw count. For any census or category column this is one of the most-used tools.

In [11]:
# how many sales rows for each city?
print(sales["city"].value_counts())

# the same, as a percentage of all rows
print((sales["city"].value_counts(normalize=True) * 100).round(1))

city
Pune      5
Mumbai    4
Delhi     3
Name: count, dtype: int64
city
Pune      41.7
Mumbai    33.3
Delhi     25.0
Name: proportion, dtype: float64


## Step 8 — filter on more than one test

Combine tests with `&` (and) or `|` (or), keeping **each test in its own
brackets** — this is the single most common beginner error. To keep rows whose
value is any of a list, use `.isin([...])` instead of a long chain of `|`.

In [12]:
# combine two tests -- keep EACH test in its own ()
big_pune = sales[(sales["city"] == "Pune") & (sales["units"] > 5)]
print("big Pune orders:")
print(big_pune)

# keep rows whose city is any value in a list, with .isin(...)
two_cities = sales[sales["city"].isin(["Mumbai", "Delhi"])]
print()
print("Mumbai or Delhi rows:", two_cities.shape)

big Pune orders:
          date  city customer  units  price
0   2024-01-03  Pune     Asha     10    250
2   2024-01-06  Pune    Meera      7    250
11  2024-01-20  Pune    Meera      6    250

Mumbai or Delhi rows: (7, 5)


### Stretch (optional) — who are the big spenders?

Skip this if you are still getting comfortable. If you want the next useful move,
here it is. The shop owner really wants spend, not just units, so we make a new
column `revenue = units * price` in one line (pandas does the multiply down the
whole column at once), then sort to see the biggest spenders on top. This is the
"which customers are the big spenders" question from the top of the session.

In [13]:
# Make a new column from two existing ones. This multiplies row by row.
sales["revenue"] = sales["units"] * sales["price"]

# Sort the table so the biggest spenders come first.
by_spend = sales.sort_values("revenue", ascending=False)

print("customers ranked by total spend (rupees):")
print(by_spend)

customers ranked by total spend (rupees):
          date    city customer  units  price  revenue
9   2024-01-17  Mumbai     Sara     13    300     3900
4   2024-01-09  Mumbai     Sara      9    300     2700
0   2024-01-03    Pune     Asha     10    250     2500
7   2024-01-14  Mumbai     Ravi      8    300     2400
2   2024-01-06    Pune    Meera      7    250     1750
11  2024-01-20    Pune    Meera      6    250     1500
5   2024-01-11    Pune     Asha      5    250     1250
1   2024-01-05  Mumbai     Ravi      4    300     1200
6   2024-01-12   Delhi   Vikram      6    180     1080
8   2024-01-15    Pune    Kiran      3    250      750
10  2024-01-18   Delhi     John      4    180      720
3   2024-01-08   Delhi     John      2    180      360


## What you just did

You read a file into a table (a DataFrame), looked at its index and columns, pulled out a
column, selected rows and columns two ways (`.loc` by name and `.iloc` by
position), and kept only the rows that passed a test. Look, select, filter: that is
most of day-to-day data work, and it is the same thing the shop owner would do by
hand in Excel, only faster and repeatable.

Next notebook: `02_group_and_join.ipynb`, where we total the sales per customer and
stitch a second sheet onto this one.